# Import Library

In [1]:
import os
import time
import pandas as pd
import io
from google import genai
from google.genai import types

# Inisialisasi API KEY

In [2]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
client = genai.Client()

# Prompt Engineering

In [3]:
print("Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)")

system_instruction = """Kamu adalah Data Engineer yang bertugas membuat dataset sintetik (dummy) untuk melatih model Natural Language Generation (Report Generator).
Tugasmu adalah membuat baris data dalam format CSV murni dengan pemisah koma (,).
HANYA BERIKAN OUTPUT DALAM FORMAT CSV tanpa markdown, tanpa penjelasan, dan dengan header yang tepat. Jangan gunakan tanda kutip di dalam teks jika akan merusak format CSV.

Kolom yang dibutuhkan:
1. linearized_data_input
2. narasi_laporan_output

Aturan Kolom 1 (linearized_data_input):
Bentuknya adalah baris data tabel yang diubah menjadi string dengan format tag khusus: [MESIN] nama aset [STATUS] Prediksi Rusak X Hari [SEVERITY] tingkat keparahan [ESTIMASI_BIAYA] jumlah biaya

Aturan Kolom 2 (narasi_laporan_output):
Paragraf laporan eksekutif formal (maksimal 2 kalimat) yang merangkum input tersebut. Ditujukan untuk dibaca oleh departemen Akuntansi/Keuangan guna persiapan mitigasi risiko dan penyiapan anggaran. NAMA ASET DARI INPUT WAJIB DITULIS ULANG SECARA PERSIS di dalam narasi.

Contoh Format Output:
linearized_data_input,narasi_laporan_output
"[MESIN] HVAC Lantai 2 [STATUS] Prediksi Rusak 15 Hari [SEVERITY] Tinggi [ESTIMASI_BIAYA] Rp 4.500.000","Berdasarkan analitik pemeliharaan bulan ini, diproyeksikan 1 unit HVAC di Lantai 2 memerlukan perbaikan dalam 15 hari ke depan dengan tingkat keparahan Tinggi. Estimasi alokasi dana perbaikan yang harus disiapkan adalah Rp 4.500.000."
"""

user_prompt = """Buatkan 50 baris data sintetik baru. 
ATURAN PALING PENTING: Gunakan variasi nama mesin/aset yang SANGAT ACAK, LUAS, dan EKSTREM. 
Jangan hanya gunakan mesin pabrik. Campurkan dengan barang sehari-hari, barang elektronik, perabotan, kendaraan, hingga barang absurd atau kode fiktif.
Contoh variasi nama aset: Pompa Air Utama, Kipas Angin Ruang HRD, Tas Hermes, Laptop Lenovo, Dispenser Galon, Alat-XYZ99, Mesin Fotokopi, Robot Vacuum, Proyektor, Kulkas Pantry, dll. 
Gunakan variasi angka sisa hari (1 sampai 30 hari). 
Gunakan variasi tingkat severity (Rendah, Sedang, Tinggi, Kritis). 
Gunakan variasi estimasi biaya (Rp 50.000 hingga Rp 35.000.000). 
Pastikan kalimat pada kolom narasi bervariasi agar model AI tidak menghafal satu gaya bahasa saja, tetapi pastikan NAMA ASET disalin persis ke dalam narasi."""

Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)


# Processing ke CSV

In [4]:
file_name = "../../data/dataset_report_generator.csv"
jumlah_loop = 10

print(f"🚀 Memulai pembuatan dataset '{file_name}'...\n")

for i in range(jumlah_loop):
    print(f"🔄 Meng-generate batch {i+1} dari {jumlah_loop}...")
    
    try:
        # Memanggil Gemini API dengan format terbaru
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.8, # Temperature 0.8 agar kalimat narasinya kaya dan bervariasi
            )
        )
        
        csv_raw_data = response.text
        
        # Membersihkan formatting markdown (```csv ... ```) jika Gemini tidak sengaja menambahkannya
        if "```csv" in csv_raw_data:
            csv_raw_data = csv_raw_data.replace("```csv", "").replace("```", "").strip()
        elif "```" in csv_raw_data:
            csv_raw_data = csv_raw_data.replace("```", "").strip()
            
        # Membaca teks CSV menjadi Pandas DataFrame
        df_sintetik = pd.read_csv(io.StringIO(csv_raw_data))
        
        # Menyimpan ke file
        if os.path.exists(file_name):
            # Jika file sudah ada, tambahkan ke baris bawahnya (append) tanpa header
            df_sintetik.to_csv(file_name, mode='a', header=False, index=False)
        else:
            # Jika file belum ada, buat baru beserta headernya
            df_sintetik.to_csv(file_name, index=False)
            
        print(f"✅ Batch {i+1} berhasil ditambahkan!")
        
        # Pause 3 detik agar tidak terkena limit API dari Google (Rate Limiting)
        time.sleep(3)
        
    except Exception as e:
        print(f"❌ Gagal memproses batch {i+1}. Error: {e}")
        print("Teks mentah dari Gemini:\n", csv_raw_data)

# 5. Mengecek Hasil Akhir
if os.path.exists(file_name):
    df_total = pd.read_csv(file_name)
    print(f"\n🎉 SELESAI! Dataset berhasil dibuat dengan total {df_total.shape[0]} baris.")
    print("\n--- 5 Baris Pertama ---")
    display(df_total.head())

🚀 Memulai pembuatan dataset '../../data/dataset_report_generator.csv'...

🔄 Meng-generate batch 1 dari 10...
❌ Gagal memproses batch 1. Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


NameError: name 'csv_raw_data' is not defined